<a href="https://colab.research.google.com/github/jessicalewinter/deep-learning-specialization/blob/main/course-notebooks/deep-architectures/Lab01_MLP_vs_CNN_aluno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 · MLP × CNN: o que a arquitetura sabe sobre imagens?

**Arquiteturas Profundas e Representação de Dados** · Especialização em Deep Learning · CIn.AI / CIn-UFPE

Neste lab você vai medir, no CIFAR-10, o efeito do **viés indutivo** da convolução:

- **Parte A (em aula, guiada, ~45 min):** carregar os dados, contar parâmetros e treinar um MLP e uma CNN.
- **Parte B (em casa):** embaralhar os pixels e ver quem sente a diferença.
- **Parte C (em casa):** a jornada da imagem, com os mapas de ativação da CNN.
- **Parte D (em casa):** o *Deep Image Prior* em miniatura: uma CNN sem treino tenta reproduzir o cachorro e o mesmo cachorro embaralhado.
- **Desafio extra (opcional, não conta para a nota):** features de uma CNN com pesos sorteados + classificador linear.

**Antes de começar**

1. *Arquivo → Salvar uma cópia no Drive* (assim você edita a sua cópia).
2. *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*.
3. Rode as células na ordem. As células **Verificação** dizem `OK` ou `ERRO`.
4. No fim, a última célula gera o seu **código de conclusão** para o formulário de entrega.

> Sem GPU disponível? O notebook percebe e usa um subconjunto de 10 mil imagens na CPU. Funciona, só fica mais lento.

## Parte A · em aula

### Passo 1 · Configuração e dados

In [1]:
# Célula 1 · Configuração
import os, time, json, hashlib, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = DEVICE == 'cuda'   # precisão mista: mais rápido na GPU T4
# Na CPU há menos imagens; mini-batches menores e mais épocas compensam o número menor de passos.
EPOCAS, BS = (5, 128) if DEVICE == 'cuda' else (8, 64)
if DEVICE == 'cuda':
    print('GPU encontrada:', torch.cuda.get_device_name(0))
else:
    print('Sem GPU: vamos usar 10 mil imagens na CPU.')
    print('Para usar GPU: Ambiente de execução > Alterar o tipo de ambiente de execução > GPU T4')

CLASSES = ['avião', 'automóvel', 'pássaro', 'gato', 'cervo', 'cachorro', 'sapo', 'cavalo', 'navio', 'caminhão']
CHECKS = {}

def check(chave, condicao, ok, erro):
    """Registra uma verificação e mostra OK ou ERRO."""
    CHECKS[chave] = bool(condicao)
    print(('OK    ' + ok) if condicao else ('ERRO  ' + erro))

GPU encontrada: Tesla T4


In [ ]:
# Célula 2 · CIFAR-10 direto na GPU
# Por padrão, baixa do servidor oficial do CIFAR-10. Se a disciplina disponibilizar um espelho, coloque aqui o
# endereço de um .npz com x_train, y_train, x_test, y_test (imagens uint8 N × 32 × 32 × 3).
ESPELHO_URL = None   # "https://drive.google.com/file/d/1fBXTy06cRH4M4fugrFSQOwqvhjOfAqDS/view?usp=sharing"

t0 = time.time()
if ESPELHO_URL:
    if 'drive.google.com' in ESPELHO_URL:
        import gdown                      # já vem no Colab; baixa arquivos grandes do Google Drive
        gdown.download(ESPELHO_URL, 'cifar10.npz', quiet=False, fuzzy=True)
    else:
        import urllib.request
        urllib.request.urlretrieve(ESPELHO_URL, 'cifar10.npz')
    d = np.load('cifar10.npz')
    tr_x, tr_y, te_x, te_y = d['x_train'], d['y_train'], d['x_test'], d['y_test']
else:
    tr = torchvision.datasets.CIFAR10(root='dados', train=True, download=True)
    te = torchvision.datasets.CIFAR10(root='dados', train=False, download=True)
    tr_x, tr_y, te_x, te_y = tr.data, np.array(tr.targets), te.data, np.array(te.targets)

N_TREINO, N_TESTE = (len(tr_x), len(te_x)) if DEVICE == 'cuda' else (10_000, 2_000)

def para_tensor(x, y, n):
    x = torch.from_numpy(x[:n]).permute(0, 3, 1, 2).float().div(255)   # N, 3, 32, 32 em [0, 1]
    return x, torch.from_numpy(np.asarray(y[:n])).long()

x_treino, y_treino = para_tensor(tr_x, tr_y, N_TREINO)
x_teste, y_teste = para_tensor(te_x, te_y, N_TESTE)

# normalização por canal (média 0, desvio 1), calculada no treino
MEDIA = x_treino.mean(dim=(0, 2, 3), keepdim=True)
DESVIO = x_treino.std(dim=(0, 2, 3), keepdim=True)
x_treino = ((x_treino - MEDIA) / DESVIO).to(DEVICE)
x_teste = ((x_teste - MEDIA) / DESVIO).to(DEVICE)
y_treino, y_teste = y_treino.to(DEVICE), y_teste.to(DEVICE)

print(f'Treino: {tuple(x_treino.shape)} · Teste: {tuple(x_teste.shape)} · {time.time() - t0:.0f}s')

 54%|█████▍    | 91.8M/170M [03:36<02:54, 451kB/s]

**Verificação.**

In [ ]:
check('dados',
      x_treino.shape[1:] == (3, 32, 32) and len(x_treino) == len(y_treino) == N_TREINO and len(x_teste) == N_TESTE,
      'dados carregados no formato N × 3 × 32 × 32',
      'formato inesperado: rode a célula 2 de novo')

In [ ]:
# Célula 3 · A imagem que atravessa a disciplina
IDX_CACHORRO = int((y_teste == 5).nonzero()[1])   # o segundo cachorro do conjunto de teste (o dos slides)

def para_imagem(x):
    """Desfaz a normalização e devolve uma imagem H × W × 3 para o matplotlib."""
    return (x.detach().cpu() * DESVIO[0] + MEDIA[0]).clamp(0, 1).permute(1, 2, 0).numpy()

fig = plt.figure(figsize=(11, 3.4))
ax = fig.add_axes([0.0, 0.05, 0.28, 0.85])
ax.imshow(para_imagem(x_teste[IDX_CACHORRO]), interpolation='nearest'); ax.axis('off')
ax.set_title(f'Nossa imagem: cachorro (teste #{IDX_CACHORRO}, 32×32)')
for c in range(10):
    i = int((y_teste == c).nonzero()[1])
    a = fig.add_axes([0.30 + (c % 5) * 0.14, 0.52 - (c // 5) * 0.47, 0.12, 0.38])
    a.imshow(para_imagem(x_teste[i]), interpolation='nearest'); a.axis('off'); a.set_title(CLASSES[c], fontsize=9)
plt.show()

plt.imsave('cachorro_cifar10.png', np.kron(para_imagem(x_teste[IDX_CACHORRO]), np.ones((16, 16, 1))))
print('Figura salva em cachorro_cifar10.png (512 × 512)')

### Passo 2 · Dois modelos e seus parâmetros

In [ ]:
# Célula 4 · MLP: a imagem vira um vetor de 3.072 números
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Flatten(),                    # 3 × 32 × 32 → 3.072
            nn.Linear(3 * 32 * 32, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )
    def forward(self, x):
        return self.rede(x)

In [ ]:
# Célula 5 · CNN pequena: três blocos conv 3×3 → ReLU → pooling
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 32 × 16 × 16
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 64 × 8 × 8
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 128 × 4 × 4
        )
        self.saida = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 10))
    def forward(self, x):
        return self.saida(self.features(x))

**Prever antes de ver.** Quantos parâmetros cada modelo tem? Qual dos dois você acha que vai acertar mais no CIFAR-10? Responda no chat antes de rodar a próxima célula.

In [ ]:
# Célula 6 · Conte os parâmetros
def count_params(model):
    """Número total de parâmetros (pesos + vieses) do modelo."""
    # TODO: complete com uma linha.
    # Dica: model.parameters() devolve os tensores de parâmetros; tensor.numel() conta os elementos de um tensor.
    raise NotImplementedError('Complete o TODO acima')

n_mlp, n_cnn = count_params(MLP()), count_params(CNN())
print(f'MLP: {n_mlp:,} parâmetros'.replace(',', '.'))
print(f'CNN: {n_cnn:,} parâmetros'.replace(',', '.'))
print(f'O MLP tem {n_mlp / n_cnn:.1f} vezes mais parâmetros que a CNN.')

**Verificação.**

In [ ]:
check('parametros', n_mlp == 1_707_274 and n_cnn == 94_538,
      'contagem de parâmetros correta',
      f'esperado MLP = 1.707.274 e CNN = 94.538; você obteve {n_mlp} e {n_cnn}')

### Passo 3 · Treinar os dois

In [ ]:
# Célula 7 · Funções de treino e avaliação
def avaliar(model, x, y, bs=1000):
    """Acurácia do modelo em (x, y)."""
    model.eval(); acertos = 0
    with torch.no_grad(), torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
        for i in range(0, len(x), bs):
            acertos += (model(x[i:i + bs]).argmax(1) == y[i:i + bs]).sum().item()
    return acertos / len(x)

def treinar(model, x_tr, y_tr, x_te, y_te, epocas=EPOCAS, bs=BS, lr=1e-3, nome='modelo'):
    """Treina com Adam e mini-batches sorteados direto da GPU. Devolve o modelo e o histórico."""
    torch.manual_seed(SEED)
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    hist = {'perda': [], 'acc_teste': []}
    for ep in range(epocas):
        model.train(); t0 = time.time()
        ordem = torch.randperm(len(x_tr), device=DEVICE)
        soma = torch.zeros((), device=DEVICE)
        for i in range(0, len(x_tr), bs):
            b = ordem[i:i + bs]
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
                perda = F.cross_entropy(model(x_tr[b]), y_tr[b])
            opt.zero_grad(set_to_none=True)
            scaler.scale(perda).backward(); scaler.step(opt); scaler.update()
            soma += perda.detach() * len(b)
        acc = avaliar(model, x_te, y_te)
        hist['perda'].append((soma / len(x_tr)).item()); hist['acc_teste'].append(acc)
        print(f'{nome} · época {ep + 1}/{epocas} · perda {hist["perda"][-1]:.3f} · acurácia de teste {acc:.1%} · {time.time() - t0:.1f}s')
    return model, hist

RESULTADOS = {}

In [ ]:
# Célula 8 · Treinar o MLP
mlp, hist_mlp = treinar(MLP(), x_treino, y_treino, x_teste, y_teste, nome='MLP')
RESULTADOS[('MLP', 'normal')] = hist_mlp['acc_teste'][-1]

In [ ]:
# Célula 9 · Treinar a CNN
cnn, hist_cnn = treinar(CNN(), x_treino, y_treino, x_teste, y_teste, nome='CNN')
RESULTADOS[('CNN', 'normal')] = hist_cnn['acc_teste'][-1]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for h, nome in [(hist_mlp, 'MLP'), (hist_cnn, 'CNN')]:
    ep = range(1, len(h['perda']) + 1)
    ax[0].plot(ep, h['perda'], marker='o', label=nome); ax[1].plot(ep, h['acc_teste'], marker='o', label=nome)
ax[0].set(title='Perda de treino', xlabel='época'); ax[1].set(title='Acurácia de teste', xlabel='época')
ax[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
for a in ax: a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.show()

**Verificação.**

In [ ]:
check('treino', ('MLP', 'normal') in RESULTADOS and ('CNN', 'normal') in RESULTADOS,
      'os dois modelos foram treinados',
      'rode as células 8 e 9 até o fim')
if RESULTADOS.get(('CNN', 'normal'), 0) > RESULTADOS.get(('MLP', 'normal'), 1):
    print('      A CNN, com 18 vezes menos parâmetros, acertou mais que o MLP.')
else:
    print('      Curioso: aqui a CNN não superou o MLP. Anote e traga para a discussão.')

In [ ]:
# Célula 10 · Para o formulário da turma
FORMULARIO_TURMA = '[link do formulário da turma]'
print(f'MLP: {RESULTADOS[("MLP", "normal")]:.1%}   CNN: {RESULTADOS[("CNN", "normal")]:.1%}')
print('Cole os dois valores no formulário:', FORMULARIO_TURMA)

## Parte B · em casa: embaralhe os pixels

Vamos trocar os pixels de lugar com **uma permutação fixa**: o pixel da posição 0 vai para uma posição sorteada, e isso vale para **todas** as imagens, de treino e de teste. Os 3 canais de cor de cada pixel andam juntos.

**Prever antes de ver:** o que acontece com a acurácia do MLP? E da CNN?

In [ ]:
# Célula 11 · A permutação
g = torch.Generator().manual_seed(7)
PERM = torch.randperm(32 * 32, generator=g).to(DEVICE)   # uma ordem nova para as 1.024 posições

def embaralhar(x):
    """Aplica PERM às posições dos pixels de um lote x (N × 3 × 32 × 32), mantendo os 3 canais juntos."""
    # TODO: achate as posições (N × 3 × 1024), reordene com PERM e volte para N × 3 × 32 × 32.
    # Dica: x.reshape(N, 3, 32 * 32)[:, :, PERM]
    raise NotImplementedError('Complete o TODO acima')


**Verificação.**

In [ ]:
_amostra = x_teste[:4]
_emb = embaralhar(_amostra)
_mesmos_valores = torch.allclose(_amostra.reshape(4, 3, -1).sort(-1).values, _emb.reshape(4, 3, -1).sort(-1).values)
_mesma_perm = torch.equal(_emb.reshape(4, 3, -1), _amostra.reshape(4, 3, -1)[:, :, PERM])
check('embaralhar', _emb.shape == _amostra.shape and _mesmos_valores and _mesma_perm and not torch.equal(_emb, _amostra),
      'embaralhar mantém os valores e usa a mesma permutação em todas as imagens',
      'confira a função embaralhar: ela deve reordenar só as posições, com PERM')

fig, ax = plt.subplots(1, 2, figsize=(5, 2.6))
ax[0].imshow(para_imagem(x_teste[IDX_CACHORRO]), interpolation='nearest'); ax[0].set_title('original')
ax[1].imshow(para_imagem(embaralhar(x_teste[IDX_CACHORRO:IDX_CACHORRO + 1])[0]), interpolation='nearest'); ax[1].set_title('embaralhada')
for a in ax: a.axis('off')
plt.show()

In [ ]:
# Célula 12 · Treinar de novo, do zero, com os pixels embaralhados
x_treino_emb, x_teste_emb = embaralhar(x_treino), embaralhar(x_teste)
mlp_emb, h_mlp_emb = treinar(MLP(), x_treino_emb, y_treino, x_teste_emb, y_teste, nome='MLP embaralhado')
cnn_emb, h_cnn_emb = treinar(CNN(), x_treino_emb, y_treino, x_teste_emb, y_teste, nome='CNN embaralhada')
RESULTADOS[('MLP', 'embaralhado')] = h_mlp_emb['acc_teste'][-1]
RESULTADOS[('CNN', 'embaralhado')] = h_cnn_emb['acc_teste'][-1]

print('\nAcurácia de teste      pixels normais   pixels embaralhados')
for m in ['MLP', 'CNN']:
    print(f'{m:<22} {RESULTADOS[(m, "normal")]:>14.1%} {RESULTADOS[(m, "embaralhado")]:>20.1%}')

fig, ax = plt.subplots(figsize=(6, 3.2))
pos = np.arange(2)
ax.bar(pos - 0.18, [RESULTADOS[(m, 'normal')] for m in ['MLP', 'CNN']], 0.36, label='pixels normais')
ax.bar(pos + 0.18, [RESULTADOS[(m, 'embaralhado')] for m in ['MLP', 'CNN']], 0.36, label='pixels embaralhados')
ax.set_xticks(pos, ['MLP', 'CNN']); ax.set_ylabel('acurácia de teste')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}')); ax.legend(); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

**Responda (3 a 5 linhas):** por que o MLP quase não muda e a CNN perde desempenho com os pixels embaralhados? Use as palavras *vizinhança* e *pesos*.

In [ ]:
# Célula 13 · Sua resposta
RESPOSTA_B = """
escreva aqui
"""

**Verificação.**

In [ ]:
_queda_mlp = RESULTADOS[('MLP', 'normal')] - RESULTADOS[('MLP', 'embaralhado')]
_queda_cnn = RESULTADOS[('CNN', 'normal')] - RESULTADOS[('CNN', 'embaralhado')]
check('tabela_b', len(RESULTADOS) == 4, 'tabela 2 × 2 completa', 'rode a célula 12 até o fim')
print(f'Queda do MLP: {_queda_mlp:.1%} · queda da CNN: {_queda_cnn:.1%}')
_texto = RESPOSTA_B.replace('escreva aqui', '').strip()
check('resposta_b', len(_texto) >= 120 and 'vizinh' in _texto.lower(),
      'resposta registrada',
      'escreva pelo menos 3 linhas e use a palavra "vizinhança"')

## Parte C · em casa, desafio: a jornada da imagem

Vamos olhar **dentro** da CNN treinada na Parte A. Um *hook* é uma função que o PyTorch chama toda vez que uma camada produz a sua saída: com ele, guardamos os mapas de ativação.

In [ ]:
# Célula 14 · Capturar os mapas de ativação com hooks
ativacoes = {}

def guardar(nome):
    def hook(modulo, entrada, saida):
        ativacoes[nome] = saida.detach().float().cpu()
    return hook

# As ReLUs da CNN estão em cnn.features[1] (1ª camada), cnn.features[4] (2ª) e cnn.features[7] (3ª).
# TODO: registre o hook guardar('camada1') na 1ª ReLU e guardar('camada3') na 3ª.
# Dica: modulo.register_forward_hook(funcao) devolve um "handle"; guarde-os na lista handles.
handles = []
raise NotImplementedError('Complete o TODO acima')

cachorro = x_teste[IDX_CACHORRO:IDX_CACHORRO + 1]
cnn.eval()
with torch.no_grad():
    cnn(cachorro);              a1, a3 = ativacoes['camada1'][0], ativacoes['camada3'][0]
    cnn(embaralhar(cachorro));  e1, e3 = ativacoes['camada1'][0], ativacoes['camada3'][0]
for h in handles: h.remove()

**Verificação.**

In [ ]:
check('hooks', a1.shape == (32, 32, 32) and a3.shape == (128, 8, 8),
      'mapas da 1ª camada (32 × 32 × 32) e da 3ª (128 × 8 × 8) capturados',
      f'formatos inesperados: {tuple(a1.shape)} e {tuple(a3.shape)}; confira os índices das ReLUs')

In [ ]:
# Célula 15 · Visualizar e salvar a figura (ela abre a aula 2)
fig, ax = plt.subplots(4, 9, figsize=(13, 6.2))
linhas = [('original · camada 1', cachorro[0], a1), ('original · camada 3', cachorro[0], a3),
          ('embaralhada · camada 1', embaralhar(cachorro)[0], e1), ('embaralhada · camada 3', embaralhar(cachorro)[0], e3)]
for r, (titulo, img, mapas) in enumerate(linhas):
    ax[r, 0].imshow(para_imagem(img), interpolation='nearest'); ax[r, 0].set_title(titulo, fontsize=9, loc='left')
    for k in range(8):
        ax[r, k + 1].imshow(mapas[k], cmap='magma', interpolation='nearest')
for a in ax.ravel(): a.axis('off')
plt.suptitle('A jornada da imagem · 8 mapas de ativação por camada', y=0.98)
plt.tight_layout(); plt.savefig('jornada_aula1.png', dpi=150); plt.show()
print('Figura salva em jornada_aula1.png')

## Parte D · em casa: a arquitetura já sabe algo antes de treinar

Na aula vimos o *Deep Image Prior* (Ulyanov, Vedaldi & Lempitsky, CVPR 2018): uma CNN **sem nenhum treino prévio** recebe um ruído fixo e é ajustada para reproduzir **uma única imagem**. Não há dataset: só essa imagem.

Vamos repetir o experimento com o nosso cachorro e com **os mesmos pixels embaralhados** pela permutação da parte B. Os valores são idênticos; só a estrutura espacial muda.

**Prever antes de ver:** qual das duas curvas de erro cai mais rápido? Leva cerca de 1 minuto na GPU (uns 40 s por imagem na CPU).

In [ ]:
# Célula 16 · Uma rede ampulheta, como no Deep Image Prior (em miniatura)
class Ampulheta(nn.Module):
    """Desce 32 → 16 → 8 → 4 com convoluções de stride 2 e sobe de volta até uma imagem 3 × 32 × 32."""
    def __init__(self, c=48):
        super().__init__()
        bloco = lambda i, o, s=1: nn.Sequential(nn.Conv2d(i, o, 3, s, 1), nn.BatchNorm2d(o), nn.LeakyReLU(0.2))
        self.desce = nn.Sequential(bloco(32, c, 2), bloco(c, c, 2), bloco(c, c, 2))
        self.sobe = nn.ModuleList([bloco(c, c), bloco(c, c), bloco(c, c)])
        self.saida = nn.Conv2d(c, 3, 1)

    def forward(self, z):
        x = self.desce(z)
        for b in self.sobe:
            x = b(F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False))
        return torch.sigmoid(self.saida(x))   # pixels em [0, 1]

# Os dois alvos, com valores em [0, 1]: o cachorro e o mesmo cachorro com os pixels embaralhados (PERM da parte B)
para_alvo = lambda x: torch.from_numpy(para_imagem(x)).permute(2, 0, 1)[None].to(DEVICE)
alvo_foto = para_alvo(x_teste[IDX_CACHORRO])
alvo_emb = para_alvo(embaralhar(x_teste[IDX_CACHORRO:IDX_CACHORRO + 1])[0])
print('Mesmos valores, outra ordem:', torch.allclose(alvo_foto.flatten().sort().values, alvo_emb.flatten().sort().values))

In [ ]:
# Célula 17 · Ajustar uma rede nova (pesos sorteados) a uma única imagem
def ajustar(alvo, passos=600, seed=0):
    """Ajusta uma Ampulheta nova para reproduzir `alvo` a partir de um ruído fixo z. Devolve as perdas e a saída no passo 100."""
    torch.manual_seed(seed)
    rede = Ampulheta().to(DEVICE)
    z = torch.rand(1, 32, 32, 32, device=DEVICE) * 0.1   # a entrada é ruído, sempre o mesmo
    opt = torch.optim.Adam(rede.parameters(), lr=0.01)
    perdas, cedo = [], None
    for passo in range(passos):
        # TODO: um passo de gradiente.
        # 1) calcule a saída da rede para z e o erro quadrático médio contra o alvo (F.mse_loss);
        # 2) zere os gradientes, faça o backward e o passo do otimizador;
        # 3) guarde perda.item() na lista perdas.
        raise NotImplementedError('Complete o TODO acima')
        if passo == 99:
            cedo = rede(z).detach()
    return np.array(perdas), cedo

t0 = time.time()
curva_foto, cedo_foto = ajustar(alvo_foto)
curva_emb, cedo_emb = ajustar(alvo_emb)
print(f'Erro final · cachorro: {curva_foto[-30:].mean():.5f} · embaralhado: {curva_emb[-30:].mean():.5f} · {time.time() - t0:.0f}s')

In [ ]:
# Célula 18 · As duas curvas e o que a rede produziu no passo 100
fig = plt.figure(figsize=(12, 4))
g = fig.add_gridspec(2, 3, width_ratios=[2.4, 1, 1])
ax = fig.add_subplot(g[:, 0])
suave = lambda c: np.convolve(c, np.ones(15) / 15, mode='valid')
ax.plot(suave(curva_foto), color='#DB1E2F', lw=2.2, label='cachorro')
ax.plot(suave(curva_emb), color='#1A1A1A', lw=2.2, label='mesmos pixels, embaralhados')
ax.set_yscale('log'); ax.set_xlabel('passos de ajuste (rede sem treino prévio)'); ax.set_ylabel('erro (MSE)')
ax.legend(frameon=False); ax.grid(alpha=.3)
for r, (nome, alvo, cedo) in enumerate([('cachorro', alvo_foto, cedo_foto), ('embaralhado', alvo_emb, cedo_emb)]):
    for c, (titulo, img) in enumerate([(f'alvo: {nome}', alvo), ('saída no passo 100', cedo)]):
        a = fig.add_subplot(g[r, c + 1])
        a.imshow(img[0].permute(1, 2, 0).cpu().numpy(), interpolation='nearest'); a.set_title(titulo, fontsize=9); a.axis('off')
plt.suptitle('Deep Image Prior em miniatura', x=0.02, ha='left')
plt.tight_layout(); plt.savefig('deep_image_prior_cachorro.png', dpi=150); plt.show()

**Verificação.**

In [ ]:
_razao = curva_emb[-30:].mean() / curva_foto[-30:].mean()
check('dip', len(curva_foto) == len(curva_emb) == 600 and _razao > 3,
      f'o cachorro terminou com erro {_razao:.0f} vezes menor que a versão embaralhada',
      f'razão {_razao:.1f}: confira o passo de gradiente na célula 17')

**Responda (2 a 4 linhas):** os valores dos pixels são os mesmos nos dois alvos. Por que a rede, **sem nenhum treino prévio**, reproduz o cachorro tão mais rápido? O que isso diz sobre a convolução?

In [ ]:
# Célula 19 · Sua resposta
RESPOSTA_D = """
escreva aqui
"""

**Verificação.**

In [ ]:
_texto_d = RESPOSTA_D.replace('escreva aqui', '').strip()
check('resposta_d', len(_texto_d) >= 80, 'resposta registrada', 'escreva pelo menos 2 linhas')

## Desafio extra · opcional: pesos sorteados + classificador linear

Saxe et al. (ICML 2011) mostraram que uma camada convolucional com **pesos sorteados**, seguida de pooling, já produz boas features. Aqui você congela uma camada assim e treina **só um classificador linear** em cima, com 5 mil imagens. Compare com o mesmo classificador nos pixels crus. Não conta para a nota; é uma prévia do *linear probing* da aula 7.

In [ ]:
# Célula 20 · Desafio extra: features aleatórias × pixels crus (cerca de 1 min)
def treinar_linear(f_tr, y_tr, f_te, y_te, epocas=30):
    torch.manual_seed(0)
    clf = nn.Linear(f_tr.shape[1], 10).to(DEVICE)
    opt = torch.optim.Adam(clf.parameters(), lr=1e-3, weight_decay=1e-4)
    for _ in range(epocas):
        ordem = torch.randperm(len(f_tr), device=DEVICE)
        for i in range(0, len(f_tr), 256):
            j = ordem[i:i + 256]
            perda = F.cross_entropy(clf(f_tr[j]), y_tr[j]); opt.zero_grad(); perda.backward(); opt.step()
    with torch.no_grad():
        return (clf(f_te).argmax(1) == y_te).float().mean().item()

torch.manual_seed(0)
aleatoria = nn.Sequential(nn.Conv2d(3, 64, 5, padding=2), nn.ReLU(), nn.MaxPool2d(4)).to(DEVICE).eval()   # 64 × 8 × 8 = 4.096 features
n_tr, n_te = 5000, 2000
with torch.no_grad():
    f_tr = aleatoria(x_treino[:n_tr]).flatten(1).float(); f_te = aleatoria(x_teste[:n_te]).flatten(1).float()
    mu, sd = f_tr.mean(0), f_tr.std(0) + 1e-6
    f_tr, f_te = (f_tr - mu) / sd, (f_te - mu) / sd
acc_pixels = treinar_linear(x_treino[:n_tr].flatten(1), y_treino[:n_tr], x_teste[:n_te].flatten(1), y_teste[:n_te])
acc_aleat = treinar_linear(f_tr, y_treino[:n_tr], f_te, y_teste[:n_te])
print(f'Classificador linear nos pixels crus (3.072 números):        {acc_pixels:.1%}')
print(f'Classificador linear em features de pesos sorteados (4.096): {acc_aleat:.1%}')
print(f'CNN treinada da parte A, para comparar:                      {RESULTADOS[("CNN", "normal")]:.1%}')

## Entrega

Preencha o seu e-mail e rode a célula. Copie o código de conclusão para o formulário de entrega, até a véspera da aula 2.

In [ ]:
# Célula 21 · Código de conclusão
EMAIL = ''   # seu e-mail, entre aspas
FORMULARIO_ENTREGA = '[link do formulário de entrega]'

faltando = [k for k in ['dados', 'parametros', 'treino', 'embaralhar', 'tabela_b', 'resposta_b', 'hooks', 'dip', 'resposta_d'] if not CHECKS.get(k)]
if not EMAIL.strip():
    print('Preencha a variável EMAIL e rode de novo.')
elif faltando:
    print('Ainda falta concluir:', ', '.join(faltando))
else:
    dados = {'email': EMAIL.strip().lower(), 'lab': 1,
             'acc': {f'{m}-{p}': round(v, 3) for (m, p), v in RESULTADOS.items()},
             'params': [n_mlp, n_cnn], 'dip': round(float(_razao), 1)}
    codigo = 'L1-' + hashlib.sha256(json.dumps(dados, sort_keys=True).encode()).hexdigest()[:10].upper()
    print('Código de conclusão:', codigo)
    print('Cole no formulário:', FORMULARIO_ENTREGA)